In [ ]:
# --- Sekcja 1: Importy i Łączenie z MLflow ---
import mlflow
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sys
import os

# Dodanie ścieżki do naszych modułów statystycznych
sys.path.append('../src')
from evaluation.bayesian_stats import BayesianComparator

# Pobranie wszystkich zakończonych biegów (runs) z MLflow
experiment_name = "CV_Breast_Cancer" # Przykładowy eksperyment
all_runs = mlflow.search_runs(experiment_names=[experiment_name])

In [ ]:
# --- Sekcja 2: Agregacja Wyników ---
# Wybieramy tylko kluczowe kolumny: nazwa modelu i metryki walidacyjne
results = all_runs[['params.model_name', 'metrics.val_mcc', 'metrics.val_auroc', 'params.fold']]
results.columns = ['Model', 'MCC', 'AUROC', 'Fold']

# Tabela zbiorcza: Średnia i odchylenie standardowe
summary_table = results.groupby('Model').agg({
    'MCC': ['mean', 'std'],
    'AUROC': ['mean', 'std']
}).round(4)

print("TABELA 1: Zbiorcze wyniki modeli (Mean ± Std)")
print(summary_table)

In [ ]:
# --- Sekcja 3: Porównanie Bayesowskie (Pairwise) ---
comparator = BayesianComparator(rope_interval=0.01)

# Porównujemy nasz flagowy TabKAN z klasycznym TabResNet
tab_kan_scores = results[results['Model'] == 'TabKAN']['MCC'].values
resnet_scores = results[results['Model'] == 'TabResNet']['MCC'].values

if len(tab_kan_scores) > 0 and len(resnet_scores) > 0:
    bayes_res = comparator.compare_models(tab_kan_scores, resnet_scores, k_folds=5)
    print(f"\nAnaliza Bayesowska: TabKAN vs TabResNet")
    print(f"P(TabKAN > TabResNet): {bayes_res['prob_a_better']:.2%}")
    print(f"P(TabResNet > TabKAN): {bayes_res['prob_b_better']:.2%}")
    print(f"P(Equivalence / ROPE): {bayes_res['prob_rope']:.2%}")
    
    comparator.plot_posterior(tab_kan_scores, resnet_scores, k_folds=5, names=("TabKAN", "TabResNet"))